# Download Video Example and YOLO Model

 At this stage we are uploading an example video, containing a sparse crowd scenario, and the CNN based YOLO v5 model. It is a popular choice for a wide range of vision AI tasks, including object detection, image segmentation, and image classification.


In [ ]:
import os
import cv2
import torch
import numpy as np
from scipy.spatial import distance
import ssl
from google.colab.patches import cv2_imshow

from google.colab import drive

# Download a sample video from Dropbox
!wget -O streamer_02.mp4 "https://www.dropbox.com/scl/fi/i8u4b714lqn1kd55qg1i3/001.mp4?rlkey=yon4uasdu9g0kd1j0ur1k03o6&st=2v10qurq&dl=1"

# Allow unverified SSL connections (for environments like Colab)
ssl._create_default_https_context = ssl._create_unverified_context

# Load the pre-trained YOLOv5s model from Ultralytics
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

--2025-06-30 09:35:00--  https://www.dropbox.com/scl/fi/i8u4b714lqn1kd55qg1i3/001.mp4?rlkey=yon4uasdu9g0kd1j0ur1k03o6&st=2v10qurq&dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.5.18, 2620:100:601d:18::a27d:512
Connecting to www.dropbox.com (www.dropbox.com)|162.125.5.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://uc072fe5ef6e792a41e1918aa9f5.dl.dropboxusercontent.com/cd/0/inline/Csnuq97bxZc9vAguBpOo-j8enXasEz64MOw5cygNAR-ydHVFz5vlXD8QU17NGX6Vgv0CzsswEcALXHsmKe_6eaercDsh2ddZyGZ0dRMPX_vpvgfijyOHsmD495P0l1EQMzNsi8P8vCNHBFNcAds6Y_UX/file?dl=1# [following]
--2025-06-30 09:35:01--  https://uc072fe5ef6e792a41e1918aa9f5.dl.dropboxusercontent.com/cd/0/inline/Csnuq97bxZc9vAguBpOo-j8enXasEz64MOw5cygNAR-ydHVFz5vlXD8QU17NGX6Vgv0CzsswEcALXHsmKe_6eaercDsh2ddZyGZ0dRMPX_vpvgfijyOHsmD495P0l1EQMzNsi8P8vCNHBFNcAds6Y_UX/file?dl=1
Resolving uc072fe5ef6e792a41e1918aa9f5.dl.dropboxusercontent.com (uc072fe5ef6e792a41e1918aa9f5.dl.dropboxusercontent.com)..

/usr/local/lib/python3.11/dist-packages/torch/hub.py:330: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


YOLOv5 🚀 2025-6-30 Python-3.11.13 torch-2.6.0+cu124 CPU

100%|██████████| 14.1M/14.1M [00:00<00:00, 126MB/s]

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


## Video Processing, People Detection and Group Counting

YOLO v5 is used to carry out the detection of individuals, while the count_grups function is used to estimate how many groups are present in a frame by considering the Euclidean distance between the centres of the bounding boxes identified by the model.

In [ ]:
# Group counting based on the Euclidean distance between bounding box centers
def count_groups(boxes, distance_threshold=100):
    if len(boxes) == 0:
        return 0

    # Calculate the center of each bounding box
    centers = [(int((box[0] + box[2]) / 2), int((box[1] + box[3]) / 2)) for box in boxes]

    # Compute pairwise distances between all centers
    dist_matrix = distance.cdist(centers, centers, 'euclidean')
    clusters = []

    # Group centers that are within the specified threshold distance
    for i, center in enumerate(centers):
        cluster = [i]
        for j in range(len(centers)):
            if dist_matrix[i][j] < distance_threshold and j != i:
                cluster.append(j)
        clusters.append(set(cluster))

    # Eliminate duplicate or overlapping clusters
    unique_clusters = []
    for cluster in clusters:
        if not any(cluster.issubset(existing) for existing in unique_clusters):
            unique_clusters.append(cluster)

    return len(unique_clusters)



# Main function to process the video and count groups frame-by-frame
def process_video(video_path, n_frames=30, distance_threshold=100, output_path=""):
    cap = cv2.VideoCapture(video_path)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # Initialize the video writer to save the annotated output
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

    frame_count = 0
    group_counts = []
    last_processed_frame = None

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Analyze every n-th frame to reduce computation
        if frame_count % n_frames == 0:
            results = model(frame)
            detections = results.xyxy[0].cpu().numpy()

            # Filter detections to only include people (targeted by YOLO as class 0)
            person_detections = [det for det in detections if int(det[5]) == 0]
            boxes = [[int(x1), int(y1), int(x2), int(y2)] for x1, y1, x2, y2, conf, cls in person_detections]

            # Count groups based on the bounding box center distances
            num_groups = count_groups(boxes, distance_threshold)
            group_counts.append(num_groups)

            # Draw bounding boxes and write group count
            for box in boxes:
                cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
            cv2.putText(frame, f'Groups: {num_groups}', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)

            last_processed_frame = frame
        else:
            # Reuse the last processed frame for skipped frames
            if last_processed_frame is not None:
                frame = last_processed_frame

        out.write(frame)
        frame_count += 1

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    return np.array(group_counts)


# Set output file path and start processing
output_video_name = "yolo_clustering-Streamer_02.mp4"
output_path = os.path.join("/content/sample_data/", output_video_name)

group_counts = process_video("streamer_02.mp4", n_frames=20, distance_threshold=100, output_path=output_path)
print(group_counts)
np.save("/content/sample_data/yolo_clustering-Streamer_02.npy", group_counts)

/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.am

[12  9  7 11  5  9 12 11 11 11 13 10 10  8 12 11  9 11 13 11  7 11 12 12 13  9  7 10 10 11 11 11  9  9 16  8  8 10 10 10 10 10  7  7  7  9  9  9  9  9 10 11 11  7 10  9 11 11 11 12  7  8  9  9 10  7 12  7  9  7 11 11 12 13  9 11 11  6 10  3  0  1  0  0]


# Descriptor

In [ ]:
"""
This python scripts describes the sequences given by the number of clusters
"""
import matplotlib
from matplotlib import pyplot as plt
import numpy as np
import os
from matplotlib.animation import PillowWriter
from matplotlib.animation import FFMpegWriter
import matplotlib.animation as animation
from collections import namedtuple

class TimeDescriptor(object):

    def __init__(self, frame_range, threshold=1, stride=1,
                 remove_redundancy=False):
        """
        time window descriptor
        :param frame_range: range of the sliding window
        :param threshold: delta for comparison
        :param stride: step of the interval selection
        :param remove_redundancy: whether or not to remove the redundancy trits
        :param mode: whether or not to return the histogram instead of the raw descriptor

        """

        self._window_size = (frame_range * 2) + 1
        self._base = 3
        self._threshold = threshold
        self._frame_range = frame_range
        self._stride = stride
        self._overlap_size = self._window_size - stride
        self._remove_redundancy = remove_redundancy

        n_bins = 3 ** (2 * self._frame_range)
        if self._remove_redundancy:
            n_to_delete = int(self._frame_range / self._stride)
            self._idx_to_delete = [self._frame_range - j * self._stride
                             for j in range(n_to_delete)]
            n_bins /= 3 ** n_to_delete
        self.bin_edges = np.arange(n_bins + 1)
        self.desc = None
        self.decimals = None
        self.hist = None

    def describe(self, x, mode=None):

        intervals = np.array(list(zip(*(x[i:] for i in range(self._window_size)))))
        self.desc = np.empty(int(intervals.shape[0]/self._stride))

        for i, interval in enumerate(intervals[::self._stride]):
            center = interval[self._frame_range]
            equal = np.less_equal(np.abs(interval - center), self._threshold)
            more = np.greater(interval, center + self._threshold)
            less = np.less(interval, center - self._threshold)
            compare = np.array([less, equal, more])
            _, compare = np.where(compare.T)

            compare = np.delete(compare, self._frame_range)
            to_delete = 0

            if self._remove_redundancy:
                n_to_delete = int(self._frame_range / self._stride)
                idx_to_delete = [self._frame_range - j * self._stride
                                for j in range(n_to_delete)]
                to_delete = int(self._frame_range / self._stride)
                compare = np.delete(compare, idx_to_delete)

            powers = np.flip(np.power(3, np.arange(
                (2 * self._frame_range) - (to_delete))), 0)

            self.desc[i] = np.dot(powers, compare.T)
        self.hist = np.histogram(self.desc, bins=self.bin_edges, density=True)

        if mode == 'h':
            return self.hist
        else:
            return self.desc


class SequenceTimeDescriptor:
    """
    This class describes a sequence of blocks of numbers with one histogram for each block.
    """

    def __init__(self, descriptor, H):
        """
        :param descriptor:
        :param H: dimension of a chunk to describe with an histogram
        """
        # pass a TimeDescriptor here
        self._descriptor = descriptor
        self._H = H
        assert (H >= self._descriptor._window_size), \
            "H is less than the window of the descriptor. Try with bigger H"


    def describe_sequence_continuous(self, X):
        """
        Describes the sequence with an histogram computed over a sliding window
        :param X: sequence to describe
        :return:
        """
        self._intervals = np.array(list(zip(*(X[i:] for i in range(self._H)))))
        self._hists = [self._descriptor.describe(ch, mode='h') for ch in self._intervals]
        return self._hists


    def plot_histogram_continuous(self, X, path, skip_frames):
        self.describe_sequence_continuous(X)
        standard_frames = (len(self._hists) - 1) * skip_frames
        number_of_frames = standard_frames
        data = self._hists

        fig = plt.figure()
        plt.bar(data[0][1][:-1], data[0][0], edgecolor='black')
        plt.ylim([0, 1])
        plt.grid(True)
        plt.gca().set_facecolor('lightgrey')

        # Aggiungi la linea di soglia
        plt.axhline(y=0.9, color = 'red', linestyle='--', linewidth=1.5)

        # Crea l'animazione
        hist_animation = animation.FuncAnimation(
            fig, update_hist, frames=number_of_frames, fargs=(data,skip_frames), repeat=False)

        # Salva l'animazione come file mp4
        writer = FFMpegWriter(fps=30, metadata=dict(artist='Me'), bitrate=1800)
        hist_animation.save(path, writer=writer)

        plt.show()
        plt.close()

    def triggers(self, X, bs, thrs):

        self.describe_sequence_continuous(X)
        watched_bins = np.array([h[0][bs] > thrs for h in self._hists])
        self.commutations = [([(not(j) and i) for (i, j) in zip(a[1:], a[:-1])])
                             for a in watched_bins.T]
        return self.commutations


def update_hist(frame,data,skip_frames):
    plt.cla()
    total_standard_frames = (len(data) - 1) * skip_frames
    if frame < total_standard_frames:
        hist_index = frame // skip_frames
    else:
        hist_index = len(data) - 1
    if hist_index < len(data):
        plt.bar(data[hist_index][1][:-1],data[hist_index][0],edgecolor = 'black')
    plt.ylim([0,1])
    plt.grid(True)
    plt.gca().set_facecolor('lightgrey')


    plt.axhline(y=0.9, color = 'red', linestyle='--', linewidth=1.5)

# Description

In [ ]:
from IPython.display import Video, display

output_data_dir = "/content/sample_data/"
output_figures_dir = "/content/sample_data/"

Parameters = namedtuple('Parameters',
                        [
                            'method',
                            'frame_range',
                            'stride',
                            'remove_redundancy',
                            'threshold',
                            'H',
                            'bin_threshold'
                        ])

# Parametri del descrittore

frame_range = 2
stride = 1
remove_redundancy = False
tol = 5
threshold = 4
Hs = 15
bin_threshold = 0.8

method_params = [
    Parameters(
        method="yolo_clustering",
        frame_range=frame_range,
        stride=stride,
        remove_redundancy=remove_redundancy,
        threshold=threshold,
        H=Hs,
        bin_threshold=bin_threshold
    )]

d = TimeDescriptor(frame_range=frame_range,
                    stride=stride,
                    threshold=threshold,
                    remove_redundancy=remove_redundancy)

s = SequenceTimeDescriptor(d, H=Hs)

clusters = np.load("/content/sample_data/yolo_clustering-Streamer_02.npy")

fig, ax = plt.subplots(figsize=(20, 12))
fig.suptitle("Group Count Reference: Streamer_02", fontsize=16, color="Gray", weight='bold')
ax.set_facecolor('#f0f0f0')
ax.set_zorder(1)
ax.set_ylabel("Triggers", fontsize=10, color="Gray", weight='bold')
ax.set_xlabel("Seconds", fontsize=10, color="Gray", weight='bold')

fps = 30
frame_skip = 20
time_between_samples = frame_skip / fps

triggers = s.triggers(clusters, [40], bin_threshold)[0]
offset = 15
triggers = np.concatenate((np.full(offset, False), triggers))
x_seconds = np.arange(len(clusters)) * time_between_samples
ax.set_xlim([0, x_seconds[-1]])
ax.set_ylim([0, 1])

ax.grid(True, linestyle='--', linewidth=0.75, alpha=0.6)
bars = ax.bar(x_seconds, triggers, width=time_between_samples, align='center', color='tomato', alpha=0.7, edgecolor='black', linewidth=0.7)

for bar in bars:
    bar.set_zorder(2)

ax.set_title("Method: Yolo", fontsize=12, color="Gray", weight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(output_figures_dir, "detections-Streamer_02.png"), bbox_inches='tight', dpi=300)
plt.show()
plt.close()

path = "/content/sample_data/Streamer_02_hist.mp4"
hist_animation = s.plot_histogram_continuous(clusters, path, frame_skip)
display(Video(path, embed=True))

# Output


In [ ]:
from moviepy.editor import VideoFileClip, clips_array, ColorClip, concatenate_videoclips
import os
from IPython.display import Video, display

detection_path = '/content/sample_data/yolo_clustering-Streamer_02.mp4'
histogram_path = '/content/sample_data/Streamer_02_hist.mp4'

output_path = '/content/sample_data'


# Video uploading
detection = VideoFileClip(detection_path)
histogram = VideoFileClip(histogram_path)

# Max duration
max_duration = max(detection.duration, histogram.duration)
if detection.duration > histogram.duration:
    black_clip = ColorClip(size=(histogram.w, histogram.h), color=(0, 0, 0), duration=(detection.duration - histogram.duration))
    histogram = concatenate_videoclips([black_clip, histogram])

video1 = detection.subclip(0, max_duration)
video2 = histogram.subclip(0, max_duration)

# Video resizing
height = min(video1.h, video2.h)
video1_resized = video1.resize(height=height)
video2_resized = video2.resize(height=height)

final_video = clips_array([[video1_resized, video2_resized]])

final_video.write_videofile(os.path.join(output_path, 'Streamer_02_output.mp4'), codec="mpeg4", fps=30)


Moviepy - Building video /content/sample_data/Streamer_02_output.mp4.
Moviepy - Writing video /content/sample_data/Streamer_02_output.mp4



Moviepy - Done !
Moviepy - video ready /content/sample_data/Streamer_02_output.mp4
